## Imports

In [1]:
import os
import pandas as pd, math
import numpy as np
from pyhive import presto
from datetime import datetime, date, timedelta
import warnings

# Ignore a specific warning by its type
warnings.filterwarnings("ignore")

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)

## Connection

In [2]:
presto_port = '80'
# username = 'airflow-user'
username = 'manoj.ravirajan@rapido.bike'

connection1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
connection2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
connection3 = presto.connect('processing-2.processing.data.production.internal', presto_port, username)
connection4 = presto.connect('processing.processing.data.production.internal', presto_port, username)

## Parameter

In [3]:
city = 'Bangalore'
date_list = [ '2024-07-19'] 
# date_list = [ '2024-04-07', '2024-04-14', '2024-04-21', '2024-04-28', '2024-05-05']
base_week = date_list[0]


city_name = city
run_date = base_week

## Dataset

## User-Defined Functions

In [4]:
def regularity_decider_v2(x,active_mean):
        
        if x<=0.2:
            return '05_rare_need'
        elif x>0.2 and x<=0.4:
            return '04_monthly'
        elif x>0.4 and x<=0.689:
            return '03_bi_weekly'
        else:
            if active_mean>2.6:
                return '01_daily'
            else:
                return '02_weekly'


## Intent Threshold
## Intent → Total RR in last 13 weeks/ No. of Active weeks
def intent_decider(x):
    
    if x<=2:
        return '03_Low'
    elif x>2 and x<=5:
        return '02_Medium'
    else:
        return '01_High'


## Consideration signal
def consideration_signal_creator(x,y,active_mean):
    
    if y<=1.5:
        if x<=0.3:
            return 'rarely_1_rr'
        elif x>0.3 and x<=0.5:
            return 'once_in_4_weeks_1_rr'
        elif x>0.5 and x<=0.75:
            return 'once_in_2_weeks_1_rr'
        else:
            return 'weekly_1_rr'
    else:
        if x<=0.3:
            return 'rarely_2+_rr'
        elif x>0.3 and x<=0.5:
            return 'once_in_4_weeks_2+_rr'
        elif x>0.5 and x<=0.75:
            return 'once_in_2_weeks_2+_rr'
        else:
            return 'weekly_2+_rr'
            

## Correction factor
def correction_factor(rr_weight,rapido_age):
    
    if rapido_age >= 90:
        week_inactive = 0
        multiplier = 1

    elif rapido_age == 0:
        week_inactive = (90-rapido_age)//7 - 1
        multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2)))

    # 7,14,21,28,...
    elif rapido_age%7 == 0:
        week_inactive = (90-rapido_age)//7 - 1
        multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2)))    
        
    else:
        week_inactive = (90-rapido_age)//7
        multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2))) 
    
    return (multiplier, 
            week_inactive, 
            (rr_weight.sum()- (week_inactive*(week_inactive+1)/2)), 
            ((rr_weight*rr_weight).sum() - (week_inactive*(week_inactive+1)*(2*week_inactive+1)/6))
           )


## Recency trend
def recency_type(days_since_last_fe):
    
    if days_since_last_fe <= 6: 
        return('Recent')
    elif days_since_last_fe <= 20:
        return('Stationary')
    else:
        return('Dormant')

## Intent trend
def intent_trend_type(intent_trend):
    
    if intent_trend <= -0.1:
        return('03_Declining')
    elif intent_trend <= 0.1:
        return('02_Stable')
    else:
        return('01_Increasing')

In [5]:
rr_data_query = f"""

    with dormant_customer as (
        
        select
            customer_id as customer,
            date_diff('day',taxi_lifetime_first_ride_date,date(run_date)) rapido_first_ride,
            ceil(cast(date_diff('day',taxi_lifetime_first_ride_date,date(run_date)) as real)/10)*10 rapido_age_seg_fr,
            
            taxi_lifetime_last_ride_date, taxi_lifetime_first_ride_date, 
            taxi_lifetime_stage, previous_taxi_lifetime_stage, taxi_income_segment,
            taxi_lifetime_rides, taxi_recency_segment, taxi_rr_active_days_last_21_days as run_taxi_rr_active_days_last_21_days,
            taxi_lifetime_last_ride_city city_name, 
            
            fe_mean_intent_type, fe_intent_trend_type, fe_potential_per_type, 
            fe_regularity_type, fe_recency_type, fe_volatility_type,
            
            gender_tag, ps_tag_link, ps_tag_auto,
            
            taxi_service_affinity, taxi_distance_preference, taxi_time_of_day_preference, taxi_day_of_week_preference
        
        from 
            datasets.iallocator_customer_segments
        where 
            date(run_date) = date('{base_week}')
            -- and taxi_lifetime_last_ride_date = date('2023-11-01')
            and taxi_recency_segment in ('RECENT')
            and taxi_lifetime_rides > 4
            and taxi_lifetime_last_ride_city in ('{city_name}')
    ),
    
    base as (
    
        select 
            a.*,
            case when coalesce(b.rr,0) <1 then 0 else 1 end as rr_retention_flag,
            case when coalesce(b.fe,0) <1 then 0 else 1 end as fe_retention_flag,
            case when coalesce(b.ao,0) <1 then 0 else 1 end as ao_retention_flag,
            case when coalesce(b.nr,0) <1 then 0 else 1 end as net_retention_flag,
            
            case when coalesce(b.nr_post_w01,0) <1 then 0 else 1 end as nr_flag_w01,
            case when coalesce(b.nr_post_w02,0) <1 then 0 else 1 end as nr_flag_w02,
            case when coalesce(b.nr_post_w03,0) <1 then 0 else 1 end as nr_flag_w03,
            case when coalesce(b.nr_post_w04,0) <1 then 0 else 1 end as nr_flag_w04,
            case when coalesce(b.nr_post_w05,0) <1 then 0 else 1 end as nr_flag_w05,
            case when coalesce(b.nr_post_w06,0) <1 then 0 else 1 end as nr_flag_w06,
            case when coalesce(b.nr_post_w07,0) <1 then 0 else 1 end as nr_flag_w07,
            case when coalesce(b.nr_post_w08,0) <1 then 0 else 1 end as nr_flag_w08,
            case when coalesce(b.nr_post_w09,0) <1 then 0 else 1 end as nr_flag_w09,
            case when coalesce(b.nr_post_w10,0) <1 then 0 else 1 end as nr_flag_w10,
            
            case when coalesce(b.rr_post_w01,0) <1 then 0 else 1 end as rr_flag_w01,
            case when coalesce(b.rr_post_w02,0) <1 then 0 else 1 end as rr_flag_w02,
            case when coalesce(b.rr_post_w03,0) <1 then 0 else 1 end as rr_flag_w03,
            case when coalesce(b.rr_post_w04,0) <1 then 0 else 1 end as rr_flag_w04,
            case when coalesce(b.rr_post_w05,0) <1 then 0 else 1 end as rr_flag_w05,
            case when coalesce(b.rr_post_w06,0) <1 then 0 else 1 end as rr_flag_w06,
            case when coalesce(b.rr_post_w07,0) <1 then 0 else 1 end as rr_flag_w07,
            case when coalesce(b.rr_post_w08,0) <1 then 0 else 1 end as rr_flag_w08,
            case when coalesce(b.rr_post_w09,0) <1 then 0 else 1 end as rr_flag_w09,
            case when coalesce(b.rr_post_w10,0) <1 then 0 else 1 end as rr_flag_w10,
            
            case when coalesce(b.fe_post_w01,0) <1 then 0 else 1 end as fe_flag_w01,
            case when coalesce(b.fe_post_w02,0) <1 then 0 else 1 end as fe_flag_w02,
            case when coalesce(b.fe_post_w03,0) <1 then 0 else 1 end as fe_flag_w03,
            case when coalesce(b.fe_post_w04,0) <1 then 0 else 1 end as fe_flag_w04,
            case when coalesce(b.fe_post_w05,0) <1 then 0 else 1 end as fe_flag_w05,
            case when coalesce(b.fe_post_w06,0) <1 then 0 else 1 end as fe_flag_w06,
            case when coalesce(b.fe_post_w07,0) <1 then 0 else 1 end as fe_flag_w07,
            case when coalesce(b.fe_post_w08,0) <1 then 0 else 1 end as fe_flag_w08,
            case when coalesce(b.fe_post_w09,0) <1 then 0 else 1 end as fe_flag_w09,
            case when coalesce(b.fe_post_w10,0) <1 then 0 else 1 end as fe_flag_w10

        from 
            dormant_customer a
        left join
            (
            select 
                customerid,
                -- sum(ao_sessions_unique_daily) ao,
                -- sum(fe_sessions_unique_daily) fe,
                -- sum(rr_sessions_unique_daily) rr,
                -- sum(net_rides_daily) nr,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then ao_sessions_unique_daily end) as ao,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then net_rides_daily end) as nr,
                
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w10,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w09,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w08,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w07,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w06,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w05,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w04,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w03,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w02,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w01,
                
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w10,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w09,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w08,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w07,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w06,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w05,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w04,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w03,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w02,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w01,
                
                
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w10,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w09,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w08,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w07,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w06,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w05,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w04,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w03,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w02,
                sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w01

            from 
                datasets.customer_rf_daily_kpi 
            where 
                date(day) > date('{base_week}') 
                and date(day) <= date('{base_week}') + interval '70' day
                and service_name in ( 'Auto', 'Auto NCR', 'Auto Pool' , 'Link', 'Bike Lite', 'Bike Metro', 'Bike Pink', 'CabEconomy', 'CabPremium')
                and city in ('{city_name}')
            group by 1
            ) b
            on a.customer=b.customerid

    ),

    rr_data as (
    
        select 
            fe.*
        from
            (
            select 
                aa.*, 
                bb.rr_LT, 
                bb.rapido_age,
                bb.rapido_age_seg
            from
                (
                select 
                    customerid,
                    sum(case when day > date_format(date('{run_date}') + interval '-91' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w13,
                    sum(case when day > date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w12,
                    sum(case when day > date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w11,
                    sum(case when day > date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w10,
                    sum(case when day > date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w09,
                    sum(case when day > date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w08,
                    sum(case when day > date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w07,
                    sum(case when day > date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w06,
                    sum(case when day > date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w05,
                    sum(case when day > date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w04,
                    sum(case when day > date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w03,
                    sum(case when day > date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w02,
                    sum(case when day > date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w01,
                    sum(case when day > date_format(date('{run_date}') + interval '-91' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_91days,
                    sum(case when day > date_format(date('{run_date}') + interval '-91' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_91days,
                    count(distinct case when day > date_format(date('{run_date}') + interval '-91' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d')  then day end) as active_rr_days_w13,
                    count(distinct case when day > date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d')  then day end) as active_rr_days_w12,
                    count(distinct case when day > date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d')  then day end) as active_rr_days_w11,
                    count(distinct case when day > date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d')  then day end) as active_rr_days_w10,
                    count(distinct case when day > date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d')  then day end) as active_rr_days_w09,
                    count(distinct case when day > date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d')  then day end) as active_rr_days_w08,
                    count(distinct case when day > date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d')  then day end) as active_rr_days_w07,
                    count(distinct case when day > date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d')  then day end) as active_rr_days_w06,
                    count(distinct case when day > date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d')  then day end) as active_rr_days_w05,
                    count(distinct case when day > date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d')  then day end) as active_rr_days_w04,
                    count(distinct case when day > date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d')  then day end) as active_rr_days_w03,
                    count(distinct case when day > date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d')  then day end) as active_rr_days_w02,
                    count(distinct case when day > date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then day end) as active_rr_days_w01,
                    day((date('{run_date}') + interval '-01' day) - date(max(day))) as days_since_last_rr
                from 
                    (
                    select 
                        customerid,day, 
                        max(fe_sessions_unique_daily) as fe_sessions_unique_daily,  
                        sum(rr_sessions_unique_daily) as rr_sessions_unique_daily
                    from 
                        datasets.customer_rf_daily_kpi a
                    join
                        base b
                        on a.customerid = b.customer
                        and 1 = 1 
                        and city = '{city_name}'
                        and rr_sessions_unique_daily >= 1 
                        and day >= date_format(date('{run_date}') + interval '-91' day , '%Y-%m-%d')
                        and day <= date_format(date('{run_date}') + interval '-1' day , '%Y-%m-%d')
                        and service_name in ( 'Auto', 'Auto NCR', 'Auto Pool' , 'Link', 'Bike Lite', 'Bike Metro', 'Bike Pink', 'CabEconomy', 'CabPremium')
                    group by customerid, day
                    )
                group by customerid
                ) aa 
            left join 
                (
                select 
                    customerid, 
                    sum(rr_sessions_unique_daily) as rr_LT , 
                    day((date('{run_date}') + interval '-01' day) - date(min(day))) as rapido_age,
                    ceil(cast(day((date('{run_date}') + interval '-01' day) - date(min(day))) as real)/10)*10 rapido_age_seg
                from 
                    datasets.customer_rf_daily_kpi a
                join
                    base b
                    on a.customerid=b.customer
                    and rr_sessions_unique_daily >= 1 
                    and city = '{city_name}'
                group by 1
                ) bb
                on aa.customerid = bb.customerid
            ) fe 
    )
    
    select 
        a.*, b.*,
        '{run_date}' as run_date
    from 
        base a
    left join
        rr_data b
        on a.customer=b.customerid
"""

try:
    data = pd.read_sql(rr_data_query, connection3)
except:
    data = pd.read_sql(rr_data_query, connection4)
    

data.to_csv('testing_jul19.csv', index=False)
## Data fetched successfully

In [5]:
data1 = pd.read_csv('testing_jul19.csv')
data1.head(2)

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date
0,658d4b387315065133bbba1e,204,210.0,2024-07-18,2023-12-28,HOOK,HOOK,UNKNOWN,21,RECENT,6,Bangalore,Low,Increasing,Low,High,Recent,Low,MALE,NPS,NPS,LINK_AUTO,LONG_DISTANCE,UNKNOWN,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,658d4b387315065133bbba1e,NaN,NaN,NaN,NaN,NaN,NaN,3.0,4.0,NaN,NaN,2.0,1.0,6.0,16.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,2.0,1.0,3.0,0.0,35.0,203.0,210.0,2024-07-19
1,65f973bb651791bf1f11726f,122,130.0,2024-07-19,2024-03-19,SUSTENANCE,INACTIVE,HIGH_INCOME,9,RECENT,3,Bangalore,Low,Stable,High,Low,Recent,High,FEMALE,UNKNOWN,UNKNOWN,UNKNOWN,MEDIUM_DISTANCE,EVENING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,65f973bb651791bf1f11726f,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,8.0,121.0,130.0,2024-07-19


In [6]:
data1.shape

(819944, 93)

In [7]:
# data1[['customer']].head(15000)

In [8]:
# data1[data1['customer'] == '61fa9f78ebef8257622bc838']

# .isin(['63058bc3ca3c325e3991febd', '650c1afb1213a91bc9edf1a0', 
#                                         '665c1b793983e81f8c129b31', '63db6fe7beabc2a4ef03caa4', '650402ee1213a9f7fca69a8e', 
#                                         '62e3fc06586b03556b8b0c68', '5cebd1fc25ee3218d4e498d9', '5fbc997190d779dee3bdecdf', 
#                                         '62670544797562573841def2', '6517c0b8513cd27442a39332', '5ca02d238c352421ea044449', 
#                                         '61dd47da85b2ba3e7f6d1b6f'])]

In [9]:
# df_test

In [10]:
# data = data1[data1['customer'].isin(['62f6016b4d8a82ead77f5e81',
# '6623445c3b0cec050d82deb4',
# '5e312c6b6f27f97bc25bbdc0',
# '5d4460d68dbbed6b8548606b',
# '653cd61e41083274309b5af0'])]
# # data.shape

In [18]:
data = data1.head(500)

## Segment derivations 

In [19]:
## .mask(data[columns_to_replace] == 0, pd.NA) replaces all 0 values in these columns with pd.NA.
columns_to_replace = ['active_rr_days_w13', 'active_rr_days_w12', 'active_rr_days_w11','active_rr_days_w10', 'active_rr_days_w09',
                      'active_rr_days_w08', 'active_rr_days_w07', 'active_rr_days_w06', 'active_rr_days_w05', 'active_rr_days_w04',
                      'active_rr_days_w03', 'active_rr_days_w02', 'active_rr_days_w01']
data[columns_to_replace] = data[columns_to_replace].mask(data[columns_to_replace] == 0, pd.NA)
data

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date
0,658d4b387315065133bbba1e,204,210.0,2024-07-18,2023-12-28,HOOK,HOOK,UNKNOWN,21,RECENT,6,Bangalore,Low,Increasing,Low,High,Recent,Low,MALE,NPS,NPS,LINK_AUTO,LONG_DISTANCE,UNKNOWN,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,658d4b387315065133bbba1e,NaN,NaN,NaN,NaN,NaN,NaN,3.0,4.0,NaN,NaN,2.0,1.0,6.0,16.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.0,NaN,NaN,2.0,1.0,3.0,0.0,35.0,203.0,210.0,2024-07-19
1,65f973bb651791bf1f11726f,122,130.0,2024-07-19,2024-03-19,SUSTENANCE,INACTIVE,HIGH_INCOME,9,RECENT,3,Bangalore,Low,Stable,High,Low,Recent,High,FEMALE,UNKNOWN,UNKNOWN,UNKNOWN,MEDIUM_DISTANCE,EVENING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,65f973bb651791bf1f11726f,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,8.0,121.0,130.0,2024-07-19
2,657281eb8af9864febe47874,224,230.0,2024-07-19,2023-12-08,COMMITTED,COMMITTED,UNKNOWN,45,RECENT,12,Bangalore,High,Declining,High,High,Recent,High,FEMALE,UNKNOWN,PS,ONLY_CAB,LONG_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,657281eb8af9864febe47874,6.0,2.0,8.0,8.0,10.0,5.0,2.0,5.0,3.0,3.0,4.0,6.0,6.0,68.0,55.0,2.0,2.0,4.0,4.0,6.0,1.0,2.0,5.0,3.0,2.0,3.0,4.0,4.0,1.0,81.0,223.0,230.0,2024-07-19
3,641da18e42419d5e7c8cfb9a,477,480.0,2024-07-19,2023-03-30,COMMITTED,COMMITTED,MEDIUM_INCOME,101,RECENT,8,Bangalore,Low,Stable,High,High,Recent,High,UNKNOWN,UNKNOWN,UNKNOWN,ONLY_AUTO,SHORT_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,641da18e42419d5e7c8cfb9a,2.0,1.0,3.0,3.0,1.0,NaN,2.0,1.0,NaN,4.0,2.0,3.0,2.0,24.0,24.0,2.0,1.0,3.0,3.0,1.0,NaN,2.0,1.0,NaN,4.0,2.0,3.0,2.0,0.0,122.0,482.0,490.0,2024-07-19
4,5d0ba5d657152c471e06c7fd,1855,1860.0,2024-07-19,2019-06-21,HOOK,DORMANT,HIGH_INCOME,142,RECENT,4,Bangalore,Low,Increasing,Low,High,Recent,Low,MALE,UNKNOWN,NPS,UNKNOWN,MEDIUM_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5d0ba5d657152c471e06c7fd,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,5.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,3.0,2.0,36.0,839.0,840.0,2024-07-19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,61b5a03161bb5197

In [20]:
raw_customer_data = data[data['customerid']!=0]
back_up = raw_customer_data.copy() ## With NA as value


fe_data2 = raw_customer_data.fillna(0)
fe_data = fe_data2[fe_data2['customerid']!=0] ## With 0 as value

In [21]:
## sum (rr_sessions_unique_daily)
fe_cols = ['rr_w13', 'rr_w12', 'rr_w11', 'rr_w10', 'rr_w09','rr_w08', 'rr_w07', 'rr_w06','rr_w05', 'rr_w04', 
           'rr_w03', 'rr_w02','rr_w01']   
## count (rr_days)
activedays_cols = ['active_rr_days_w13', 'active_rr_days_w12', 'active_rr_days_w11', 'active_rr_days_w10', 'active_rr_days_w09',
                   'active_rr_days_w08', 'active_rr_days_w07', 'active_rr_days_w06', 'active_rr_days_w05', 'active_rr_days_w04', 
                   'active_rr_days_w03','active_rr_days_w02', 'active_rr_days_w01']

## Signal Generation

In [22]:
## rr_mean_intent

## Total rr/no. of active weeks

fe_data['rr_mean_intent'] =  back_up[fe_cols].mean(axis=1) 
fe_data['rr_mean_intent_final'] = fe_data['rr_mean_intent'].apply(lambda x: intent_decider(x))
fe_data

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date,rr_mean_intent,rr_mean_intent_final
0,658d4b387315065133bbba1e,204,210.0,2024-07-18,2023-12-28,HOOK,HOOK,UNKNOWN,21,RECENT,6,Bangalore,Low,Increasing,Low,High,Recent,Low,MALE,NPS,NPS,LINK_AUTO,LONG_DISTANCE,UNKNOWN,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,658d4b387315065133bbba1e,0.0,0.0,0.0,0.0,0.0,0.0,3.0,4.0,0.0,0.0,2.0,1.0,6.0,16.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,2.0,1.0,3.0,0.0,35.0,203.0,210.0,2024-07-19,3.200000,02_Medium
1,65f973bb651791bf1f11726f,122,130.0,2024-07-19,2024-03-19,SUSTENANCE,INACTIVE,HIGH_INCOME,9,RECENT,3,Bangalore,Low,Stable,High,Low,Recent,High,FEMALE,UNKNOWN,UNKNOWN,UNKNOWN,MEDIUM_DISTANCE,EVENING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,65f973bb651791bf1f11726f,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,8.0,121.0,130.0,2024-07-19,1.000000,03_Low
2,657281eb8af9864febe47874,224,230.0,2024-07-19,2023-12-08,COMMITTED,COMMITTED,UNKNOWN,45,RECENT,12,Bangalore,High,Declining,High,High,Recent,High,FEMALE,UNKNOWN,PS,ONLY_CAB,LONG_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,657281eb8af9864febe47874,6.0,2.0,8.0,8.0,10.0,5.0,2.0,5.0,3.0,3.0,4.0,6.0,6.0,68.0,55.0,2.0,2.0,4.0,4.0,6.0,1.0,2.0,5.0,3.0,2.0,3.0,4.0,4.0,1.0,81.0,223.0,230.0,2024-07-19,5.230769,01_High
3,641da18e42419d5e7c8cfb9a,477,480.0,2024-07-19,2023-03-30,COMMITTED,COMMITTED,MEDIUM_INCOME,101,RECENT,8,Bangalore,Low,Stable,High,High,Recent,High,UNKNOWN,UNKNOWN,UNKNOWN,ONLY_AUTO,SHORT_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,641da18e42419d5e7c8cfb9a,2.0,1.0,3.0,3.0,1.0,0.0,2.0,1.0,0.0,4.0,2.0,3.0,2.0,24.0,24.0,2.0,1.0,3.0,3.0,1.0,0.0,2.0,1.0,0.0,4.0,2.0,3.0,2.0,0.0,122.0,482.0,490.0,2024-07-19,2.181818,02_Medium
4,5d0ba5d657152c471e06c7fd,1855,1860.0,2024-07-19,2019-06-21,HOOK,DORMANT,HIGH_INCOME,142,RECENT,4,Bangalore,Low,Increasing,Low,High,Recent,Low,MALE,UNKNOWN,NPS,UNKNOWN,MEDIUM_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5d0ba5d657152c471e06c7fd,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,5.0,7.0,7.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,2.0,36.0,839.0,840.0,2024-07-19,2.333333,02_Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..

In [23]:
rr_weight = np.array([1,1,3,4,5,6,7,8,9,10,11,12,13])

In [24]:
## rr_intent_multiplier
fe_data['rr_intent_multiplier'] = fe_data['rapido_age_seg_fr'].apply(lambda x: correction_factor(rr_weight,x)[0])
fe_data

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date,rr_mean_intent,rr_mean_intent_final,rr_intent_multiplier
0,658d4b387315065133bbba1e,204,210.0,2024-07-18,2023-12-28,HOOK,HOOK,UNKNOWN,21,RECENT,6,Bangalore,Low,Increasing,Low,High,Recent,Low,MALE,NPS,NPS,LINK_AUTO,LONG_DISTANCE,UNKNOWN,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,658d4b387315065133bbba1e,0.0,0.0,0.0,0.0,0.0,0.0,3.0,4.0,0.0,0.0,2.0,1.0,6.0,16.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,2.0,1.0,3.0,0.0,35.0,203.0,210.0,2024-07-19,3.200000,02_Medium,1.0
1,65f973bb651791bf1f11726f,122,130.0,2024-07-19,2024-03-19,SUSTENANCE,INACTIVE,HIGH_INCOME,9,RECENT,3,Bangalore,Low,Stable,High,Low,Recent,High,FEMALE,UNKNOWN,UNKNOWN,UNKNOWN,MEDIUM_DISTANCE,EVENING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,65f973bb651791bf1f11726f,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,8.0,121.0,130.0,2024-07-19,1.000000,03_Low,1.0
2,657281eb8af9864febe47874,224,230.0,2024-07-19,2023-12-08,COMMITTED,COMMITTED,UNKNOWN,45,RECENT,12,Bangalore,High,Declining,High,High,Recent,High,FEMALE,UNKNOWN,PS,ONLY_CAB,LONG_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,657281eb8af9864febe47874,6.0,2.0,8.0,8.0,10.0,5.0,2.0,5.0,3.0,3.0,4.0,6.0,6.0,68.0,55.0,2.0,2.0,4.0,4.0,6.0,1.0,2.0,5.0,3.0,2.0,3.0,4.0,4.0,1.0,81.0,223.0,230.0,2024-07-19,5.230769,01_High,1.0
3,641da18e42419d5e7c8cfb9a,477,480.0,2024-07-19,2023-03-30,COMMITTED,COMMITTED,MEDIUM_INCOME,101,RECENT,8,Bangalore,Low,Stable,High,High,Recent,High,UNKNOWN,UNKNOWN,UNKNOWN,ONLY_AUTO,SHORT_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,641da18e42419d5e7c8cfb9a,2.0,1.0,3.0,3.0,1.0,0.0,2.0,1.0,0.0,4.0,2.0,3.0,2.0,24.0,24.0,2.0,1.0,3.0,3.0,1.0,0.0,2.0,1.0,0.0,4.0,2.0,3.0,2.0,0.0,122.0,482.0,490.0,2024-07-19,2.181818,02_Medium,1.0
4,5d0ba5d657152c471e06c7fd,1855,1860.0,2024-07-19,2019-06-21,HOOK,DORMANT,HIGH_INCOME,142,RECENT,4,Bangalore,Low,Increasing,Low,High,Recent,Low,MALE,UNKNOWN,NPS,UNKNOWN,MEDIUM_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5d0ba5d657152c471e06c7fd,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,5.0,7.0,7.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,2.0,36.0,839.0,840.0,2024-07-19,2.333333,02_Medium,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,.

In [25]:
## 
fe = fe_data[fe_cols]
fe

,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01
0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,4.0,0.0,0.0,2.0,1.0,6.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,6.0,2.0,8.0,8.0,10.0,5.0,2.0,5.0,3.0,3.0,4.0,6.0,6.0
3,2.0,1.0,3.0,3.0,1.0,0.0,2.0,1.0,0.0,4.0,2.0,3.0,2.0
4,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0.0,2.0,0.0,1.0,1.0,1.0,3.0,3.0,0.0,2.0,3.0,1.0,1.0
496,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,8.0,1.0,1.0
497,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
498,0.0,0.0,3.0,3.0,1.0,0.0,0.0,2.0,2.0,0.0,0.0,0.0,3.0


In [26]:
fe = fe.ge(1.0).astype(int)
fe

,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01
0,0,0,0,0,0,0,1,1,0,0,1,1,1
1,0,0,0,0,0,0,0,0,0,0,0,0,1
2,1,1,1,1,1,1,1,1,1,1,1,1,1
3,1,1,1,1,1,0,1,1,0,1,1,1,1
4,0,0,0,0,0,1,1,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0,1,0,1,1,1,1,1,0,1,1,1,1
496,0,0,0,0,0,0,0,0,0,1,1,1,1
497,0,0,0,0,0,0,0,0,0,0,0,0,1
498,0,0,1,1,1,0,0,1,1,0,0,0,1


In [27]:
## rr_regularity

# This multiplies the sum of the weighted values by 1/91 to normalize the result.
# rr_w13 to rr_w01 || rr_weight 1 to 13
# np.array(fe) * rr_weight performs element-wise multiplication of the binary values in fe with the corresponding weights in rr_weight.
# The .sum(axis=1) method sums these weighted values across each row.
# The resulting sums are divided by 91.

fe_data['rr_regularity'] = (1/90)*(np.array(fe)*rr_weight).sum(axis=1)
fe_data['rr_regularity_scaled'] = fe_data['rr_regularity']*fe_data['rr_intent_multiplier']
fe_data

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date,rr_mean_intent,rr_mean_intent_final,rr_intent_multiplier,rr_regularity,rr_regularity_scaled
0,658d4b387315065133bbba1e,204,210.0,2024-07-18,2023-12-28,HOOK,HOOK,UNKNOWN,21,RECENT,6,Bangalore,Low,Increasing,Low,High,Recent,Low,MALE,NPS,NPS,LINK_AUTO,LONG_DISTANCE,UNKNOWN,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,658d4b387315065133bbba1e,0.0,0.0,0.0,0.0,0.0,0.0,3.0,4.0,0.0,0.0,2.0,1.0,6.0,16.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,2.0,1.0,3.0,0.0,35.0,203.0,210.0,2024-07-19,3.200000,02_Medium,1.0,0.566667,0.566667
1,65f973bb651791bf1f11726f,122,130.0,2024-07-19,2024-03-19,SUSTENANCE,INACTIVE,HIGH_INCOME,9,RECENT,3,Bangalore,Low,Stable,High,Low,Recent,High,FEMALE,UNKNOWN,UNKNOWN,UNKNOWN,MEDIUM_DISTANCE,EVENING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,65f973bb651791bf1f11726f,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,8.0,121.0,130.0,2024-07-19,1.000000,03_Low,1.0,0.144444,0.144444
2,657281eb8af9864febe47874,224,230.0,2024-07-19,2023-12-08,COMMITTED,COMMITTED,UNKNOWN,45,RECENT,12,Bangalore,High,Declining,High,High,Recent,High,FEMALE,UNKNOWN,PS,ONLY_CAB,LONG_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,657281eb8af9864febe47874,6.0,2.0,8.0,8.0,10.0,5.0,2.0,5.0,3.0,3.0,4.0,6.0,6.0,68.0,55.0,2.0,2.0,4.0,4.0,6.0,1.0,2.0,5.0,3.0,2.0,3.0,4.0,4.0,1.0,81.0,223.0,230.0,2024-07-19,5.230769,01_High,1.0,1.000000,1.000000
3,641da18e42419d5e7c8cfb9a,477,480.0,2024-07-19,2023-03-30,COMMITTED,COMMITTED,MEDIUM_INCOME,101,RECENT,8,Bangalore,Low,Stable,High,High,Recent,High,UNKNOWN,UNKNOWN,UNKNOWN,ONLY_AUTO,SHORT_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,641da18e42419d5e7c8cfb9a,2.0,1.0,3.0,3.0,1.0,0.0,2.0,1.0,0.0,4.0,2.0,3.0,2.0,24.0,24.0,2.0,1.0,3.0,3.0,1.0,0.0,2.0,1.0,0.0,4.0,2.0,3.0,2.0,0.0,122.0,482.0,490.0,2024-07-19,2.181818,02_Medium,1.0,0.833333,0.833333
4,5d0ba5d657152c471e06c7fd,1855,1860.0,2024-07-19,2019-06-21,HOOK,DORMANT,HIGH_INCOME,142,RECENT,4,Bangalore,Low,Increasing,Low,High,Recent,Low,MALE,UNKNOWN,NPS,UNKNOWN,MEDIUM_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5d0ba5d657152c471e06c7fd,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,5.0,7.0,7.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,2.0,36.0,839.0,840.0,2024-07-19,2.333333,02_Medium,1.0,0.288889,0.288889
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,

In [28]:
back_up[activedays_cols]

,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01
0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.0,NaN,NaN,2.0,1.0,3.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
2,2.0,2.0,4.0,4.0,6.0,1.0,2.0,5.0,3.0,2.0,3.0,4.0,4.0
3,2.0,1.0,3.0,3.0,1.0,NaN,2.0,1.0,NaN,4.0,2.0,3.0,2.0
4,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,NaN,2.0,NaN,1.0,1.0,1.0,3.0,2.0,NaN,1.0,2.0,1.0,1.0
496,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,5.0,1.0,1.0
497,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
498,NaN,NaN,3.0,3.0,1.0,NaN,NaN,2.0,2.0,NaN,NaN,NaN,2.0


In [29]:
## active_days_mean

## Total no. of active days/no. of active weeks in last 13 weeks

fe_data['active_days_mean'] =  back_up[activedays_cols].mean(axis=1) 

fe_data[['active_rr_days_w13', 'active_rr_days_w12', 'active_rr_days_w11', 'active_rr_days_w10', 'active_rr_days_w09',
                   'active_rr_days_w08', 'active_rr_days_w07', 'active_rr_days_w06', 'active_rr_days_w05', 'active_rr_days_w04', 
                   'active_rr_days_w03','active_rr_days_w02', 'active_rr_days_w01', 'active_days_mean']]

,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,active_days_mean
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,2.0,1.0,3.0,1.800000
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.000000
2,2.0,2.0,4.0,4.0,6.0,1.0,2.0,5.0,3.0,2.0,3.0,4.0,4.0,3.230769
3,2.0,1.0,3.0,3.0,1.0,0.0,2.0,1.0,0.0,4.0,2.0,3.0,2.0,2.181818
4,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,1.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0.0,2.0,0.0,1.0,1.0,1.0,3.0,2.0,0.0,1.0,2.0,1.0,1.0,1.500000
496,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,5.0,1.0,1.0,3.000000
497,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.000000
498,0.0,0.0,3.0,3.0,1.0,0.0,0.0,2.0,2.0,0.0,0.0,0.0,2.0,2.166667


In [30]:
fe_data['rr_regularity_final'] = fe_data.apply(lambda x: regularity_decider_v2(x['rr_regularity_scaled'],x['active_days_mean']),axis=1)
fe_data['reg_intent'] = fe_data.rr_regularity_final+'_'+fe_data.rr_mean_intent_final

In [31]:
fe_data

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date,rr_mean_intent,rr_mean_intent_final,rr_intent_multiplier,rr_regularity,rr_regularity_scaled,active_days_mean,rr_regularity_final,reg_intent
0,658d4b387315065133bbba1e,204,210.0,2024-07-18,2023-12-28,HOOK,HOOK,UNKNOWN,21,RECENT,6,Bangalore,Low,Increasing,Low,High,Recent,Low,MALE,NPS,NPS,LINK_AUTO,LONG_DISTANCE,UNKNOWN,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,658d4b387315065133bbba1e,0.0,0.0,0.0,0.0,0.0,0.0,3.0,4.0,0.0,0.0,2.0,1.0,6.0,16.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,2.0,1.0,3.0,0.0,35.0,203.0,210.0,2024-07-19,3.200000,02_Medium,1.0,0.566667,0.566667,1.800000,03_bi_weekly,03_bi_weekly_02_Medium
1,65f973bb651791bf1f11726f,122,130.0,2024-07-19,2024-03-19,SUSTENANCE,INACTIVE,HIGH_INCOME,9,RECENT,3,Bangalore,Low,Stable,High,Low,Recent,High,FEMALE,UNKNOWN,UNKNOWN,UNKNOWN,MEDIUM_DISTANCE,EVENING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,65f973bb651791bf1f11726f,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,8.0,121.0,130.0,2024-07-19,1.000000,03_Low,1.0,0.144444,0.144444,1.000000,05_rare_need,05_rare_need_03_Low
2,657281eb8af9864febe47874,224,230.0,2024-07-19,2023-12-08,COMMITTED,COMMITTED,UNKNOWN,45,RECENT,12,Bangalore,High,Declining,High,High,Recent,High,FEMALE,UNKNOWN,PS,ONLY_CAB,LONG_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,657281eb8af9864febe47874,6.0,2.0,8.0,8.0,10.0,5.0,2.0,5.0,3.0,3.0,4.0,6.0,6.0,68.0,55.0,2.0,2.0,4.0,4.0,6.0,1.0,2.0,5.0,3.0,2.0,3.0,4.0,4.0,1.0,81.0,223.0,230.0,2024-07-19,5.230769,01_High,1.0,1.000000,1.000000,3.230769,01_daily,01_daily_01_High
3,641da18e42419d5e7c8cfb9a,477,480.0,2024-07-19,2023-03-30,COMMITTED,COMMITTED,MEDIUM_INCOME,101,RECENT,8,Bangalore,Low,Stable,High,High,Recent,High,UNKNOWN,UNKNOWN,UNKNOWN,ONLY_AUTO,SHORT_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,641da18e42419d5e7c8cfb9a,2.0,1.0,3.0,3.0,1.0,0.0,2.0,1.0,0.0,4.0,2.0,3.0,2.0,24.0,24.0,2.0,1.0,3.0,3.0,1.0,0.0,2.0,1.0,0.0,4.0,2.0,3.0,2.0,0.0,122.0,482.0,490.0,2024-07-19,2.181818,02_Medium,1.0,0.833333,0.833333,2.181818,02_weekly,02_weekly_02_Medium
4,5d0ba5d657152c471e06c7fd,1855,1860.0,2024-07-19,2019-06-21,HOOK,DORMANT,HIGH_INCOME,142,RECENT,4,Bangalore,Low,Increasing,Low,High,Recent,Low,MALE,UNKNOWN,NPS,UNKNOWN,MEDIUM_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5d0ba5d657152c471e06c7fd,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,5.0,7.0,7.0,0.0,0.0,0.0,0.0,0.0,

## Final df

In [32]:
concatenated_df = fe_data.copy()
concatenated_df['city'] = city_name
concatenated_df['reg_intent'] = concatenated_df.rr_regularity_final + '_' + concatenated_df.rr_mean_intent_final

In [33]:
concatenated_df

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date,rr_mean_intent,rr_mean_intent_final,rr_intent_multiplier,rr_regularity,rr_regularity_scaled,active_days_mean,rr_regularity_final,reg_intent,city
0,658d4b387315065133bbba1e,204,210.0,2024-07-18,2023-12-28,HOOK,HOOK,UNKNOWN,21,RECENT,6,Bangalore,Low,Increasing,Low,High,Recent,Low,MALE,NPS,NPS,LINK_AUTO,LONG_DISTANCE,UNKNOWN,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,658d4b387315065133bbba1e,0.0,0.0,0.0,0.0,0.0,0.0,3.0,4.0,0.0,0.0,2.0,1.0,6.0,16.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,2.0,1.0,3.0,0.0,35.0,203.0,210.0,2024-07-19,3.200000,02_Medium,1.0,0.566667,0.566667,1.800000,03_bi_weekly,03_bi_weekly_02_Medium,Bangalore
1,65f973bb651791bf1f11726f,122,130.0,2024-07-19,2024-03-19,SUSTENANCE,INACTIVE,HIGH_INCOME,9,RECENT,3,Bangalore,Low,Stable,High,Low,Recent,High,FEMALE,UNKNOWN,UNKNOWN,UNKNOWN,MEDIUM_DISTANCE,EVENING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,65f973bb651791bf1f11726f,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,8.0,121.0,130.0,2024-07-19,1.000000,03_Low,1.0,0.144444,0.144444,1.000000,05_rare_need,05_rare_need_03_Low,Bangalore
2,657281eb8af9864febe47874,224,230.0,2024-07-19,2023-12-08,COMMITTED,COMMITTED,UNKNOWN,45,RECENT,12,Bangalore,High,Declining,High,High,Recent,High,FEMALE,UNKNOWN,PS,ONLY_CAB,LONG_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,657281eb8af9864febe47874,6.0,2.0,8.0,8.0,10.0,5.0,2.0,5.0,3.0,3.0,4.0,6.0,6.0,68.0,55.0,2.0,2.0,4.0,4.0,6.0,1.0,2.0,5.0,3.0,2.0,3.0,4.0,4.0,1.0,81.0,223.0,230.0,2024-07-19,5.230769,01_High,1.0,1.000000,1.000000,3.230769,01_daily,01_daily_01_High,Bangalore
3,641da18e42419d5e7c8cfb9a,477,480.0,2024-07-19,2023-03-30,COMMITTED,COMMITTED,MEDIUM_INCOME,101,RECENT,8,Bangalore,Low,Stable,High,High,Recent,High,UNKNOWN,UNKNOWN,UNKNOWN,ONLY_AUTO,SHORT_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,641da18e42419d5e7c8cfb9a,2.0,1.0,3.0,3.0,1.0,0.0,2.0,1.0,0.0,4.0,2.0,3.0,2.0,24.0,24.0,2.0,1.0,3.0,3.0,1.0,0.0,2.0,1.0,0.0,4.0,2.0,3.0,2.0,0.0,122.0,482.0,490.0,2024-07-19,2.181818,02_Medium,1.0,0.833333,0.833333,2.181818,02_weekly,02_weekly_02_Medium,Bangalore
4,5d0ba5d657152c471e06c7fd,1855,1860.0,2024-07-19,2019-06-21,HOOK,DORMANT,HIGH_INCOME,142,RECENT,4,Bangalore,Low,Increasing,Low,High,Recent,Low,MALE,UNKNOWN,NPS,UNKNOWN,MEDIUM_DISTANCE,MORNING_PEAK,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5d0ba5d657152c471e06c7fd,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0

In [35]:
concatenated_df[['customer', 'rr_regularity', 'active_days_mean', 
                 'rr_mean_intent', 'rr_regularity_scaled', 'rr_intent_multiplier', 'rr_regularity_final', 'reg_intent'
                ]].head(5)

,customer,rr_regularity,active_days_mean,rr_mean_intent,rr_regularity_scaled,rr_intent_multiplier,rr_regularity_final,reg_intent
0,658d4b387315065133bbba1e,0.566667,1.800000,3.200000,0.566667,1.0,03_bi_weekly,03_bi_weekly_02_Medium
1,65f973bb651791bf1f11726f,0.144444,1.000000,1.000000,0.144444,1.0,05_rare_need,05_rare_need_03_Low
2,657281eb8af9864febe47874,1.000000,3.230769,5.230769,1.000000,1.0,01_daily,01_daily_01_High
3,641da18e42419d5e7c8cfb9a,0.833333,2.181818,2.181818,0.833333,1.0,02_weekly,02_weekly_02_Medium
4,5d0ba5d657152c471e06c7fd,0.288889,1.666667,2.333333,0.288889,1.0,04_monthly,04_monthly_02_Medium


## Validations Function

In [36]:
def validation_query(run_date, city_name, base_week):
    
    validation_query = f"""
  
       select
            -- distinct
            -- taxi_ltr_segment,
            customer_id,
            taxi_regularity_segment,
            taxi_intent_segment
            
        from 
            datasets.iallocator_customer_segments
        where 
            run_date = '2024-07-19'
            and taxi_lifetime_rides > 4
            and taxi_lifetime_last_ride_city =  '{city_name}'
            and taxi_recency_segment != 'INACTIVE'
    """

    try:
        validation_data = pd.read_sql(validation_query, connection3)
    except:
        validation_data = pd.read_sql(validation_query, connection4)

    return validation_data

In [37]:
validation_df=pd.DataFrame()

for i in date_list:
    
    print(i)
    df_validation = validation_query(i, city, base_week)

2024-07-19


In [38]:
df_validation.shape

(2515836, 3)

In [39]:
df_validation[df_validation['customer_id'] == '61fa9f78ebef8257622bc838'].head(1)

,customer_id,taxi_regularity_segment,taxi_intent_segment
1218836,61fa9f78ebef8257622bc838,DAILY,HIGH


In [ ]:
# df2 = df_validation[[ 'customer_id', 'taxi_consistency_rr_correction_factor',
#                      'taxi_consistency_rr_regularity_90', 'taxi_consistency_rr_regularity_scaled_90', 
#                'taxi_consistency_rr_actv_days_mean_90', 'taxi_intent_rr_actv_mean_90', 'taxi_regularity_segment', 
#                'taxi_intent_segment'
#               ]]

In [40]:
df2 = df_validation

In [ ]:
concatenated_df.head(2)

In [41]:
df1 = concatenated_df[['customer', 'rr_intent_multiplier', 'rr_regularity', 'rr_regularity_scaled',  'rr_mean_intent', 'active_days_mean',
                 'rr_regularity_final', 'rr_mean_intent_final', 'reg_intent'
                ]]

In [42]:
df_merge = pd.merge(df1, df2, how='left', left_on='customer', right_on = 'customer_id')
df_merge

,customer,rr_intent_multiplier,rr_regularity,rr_regularity_scaled,rr_mean_intent,active_days_mean,rr_regularity_final,rr_mean_intent_final,reg_intent,customer_id,taxi_regularity_segment,taxi_intent_segment
0,658d4b387315065133bbba1e,1.0,0.566667,0.566667,3.200000,1.800000,03_bi_weekly,02_Medium,03_bi_weekly_02_Medium,658d4b387315065133bbba1e,BI_WEEKLY,MEDIUM
1,65f973bb651791bf1f11726f,1.0,0.144444,0.144444,1.000000,1.000000,05_rare_need,03_Low,05_rare_need_03_Low,65f973bb651791bf1f11726f,RARE_NEED,MEDIUM
2,657281eb8af9864febe47874,1.0,1.000000,1.000000,5.230769,3.230769,01_daily,01_High,01_daily_01_High,657281eb8af9864febe47874,DAILY,HIGH
3,641da18e42419d5e7c8cfb9a,1.0,0.833333,0.833333,2.181818,2.181818,02_weekly,02_Medium,02_weekly_02_Medium,641da18e42419d5e7c8cfb9a,BI_WEEKLY,MEDIUM
4,5d0ba5d657152c471e06c7fd,1.0,0.288889,0.288889,2.333333,1.666667,04_monthly,02_Medium,04_monthly_02_Medium,5d0ba5d657152c471e06c7fd,MONTHLY,MEDIUM
...,...,...,...,...,...,...,...,...,...,...,...,...
475,61b5a03161bb5197a8cb8a74,1.0,0.855556,0.855556,1.800000,1.500000,02_weekly,03_Low,02_weekly_03_Low,61b5a03161bb5197a8cb8a74,WEEKLY,LOW
476,622f3cbf9583b33f7306ccdc,1.0,0.511111,0.511111,4.750000,3.000000,03_bi_weekly,02_Medium,03_bi_weekly_02_Medium,622f3cbf9583b33f7306ccdc,WEEKLY,MEDIUM
477,62be1b7217af9a31fa66b989,1.0,0.144444,0.144444,1.000000,1.000000,05_rare_need,03_Low,05_rare_need_03_Low,62be1b7217af9a31fa66b989,MONTHLY,LOW
478,5ab4a090b63e19155e7d4b3c,1.0,0.466667,0.466667,2.333333,2.166667,03_bi_weekly,02_Medium,03_bi_weekly_02_Medium,5ab4a090b63e19155e7d4b3c,BI_WEEKLY,MEDIUM


In [43]:
df_merge.head(5)

,customer,rr_intent_multiplier,rr_regularity,rr_regularity_scaled,rr_mean_intent,active_days_mean,rr_regularity_final,rr_mean_intent_final,reg_intent,customer_id,taxi_regularity_segment,taxi_intent_segment
0,658d4b387315065133bbba1e,1.0,0.566667,0.566667,3.200000,1.800000,03_bi_weekly,02_Medium,03_bi_weekly_02_Medium,658d4b387315065133bbba1e,BI_WEEKLY,MEDIUM
1,65f973bb651791bf1f11726f,1.0,0.144444,0.144444,1.000000,1.000000,05_rare_need,03_Low,05_rare_need_03_Low,65f973bb651791bf1f11726f,RARE_NEED,MEDIUM
2,657281eb8af9864febe47874,1.0,1.000000,1.000000,5.230769,3.230769,01_daily,01_High,01_daily_01_High,657281eb8af9864febe47874,DAILY,HIGH
3,641da18e42419d5e7c8cfb9a,1.0,0.833333,0.833333,2.181818,2.181818,02_weekly,02_Medium,02_weekly_02_Medium,641da18e42419d5e7c8cfb9a,BI_WEEKLY,MEDIUM
4,5d0ba5d657152c471e06c7fd,1.0,0.288889,0.288889,2.333333,1.666667,04_monthly,02_Medium,04_monthly_02_Medium,5d0ba5d657152c471e06c7fd,MONTHLY,MEDIUM


In [44]:
# df_merge[['customer', 'rr_intent_multiplier', 'taxi_consistency_rr_correction_factor',	
#           'rr_regularity',	'taxi_consistency_rr_regularity_90',
#           'rr_regularity_scaled',	'taxi_consistency_rr_regularity_scaled_90',
#           'rr_mean_intent', 'taxi_intent_rr_actv_mean_90', 'active_days_mean',	'taxi_consistency_rr_actv_days_mean_90',
#           'rr_regularity_final', 'rr_mean_intent_final',  'taxi_regularity_segment', 'taxi_intent_segment'
#          ]]

In [45]:
df_merge.customer.nunique()

480

In [46]:
df_merge.customer_id.nunique()

480

In [47]:
df_overlap = df_merge\
.groupby(['rr_regularity_final', 'rr_mean_intent_final',  'taxi_regularity_segment', 'taxi_intent_segment'])\
.agg(customers = ('customer', 'nunique'))\
.reset_index()

df_overlap

,rr_regularity_final,rr_mean_intent_final,taxi_regularity_segment,taxi_intent_segment,customers
0,01_daily,01_High,DAILY,HIGH,58
1,01_daily,01_High,DAILY,MEDIUM,3
2,01_daily,01_High,WEEKLY,HIGH,1
3,01_daily,02_Medium,DAILY,HIGH,1
4,01_daily,02_Medium,DAILY,MEDIUM,17
5,01_daily,02_Medium,WEEKLY,MEDIUM,2
6,02_weekly,01_High,DAILY,HIGH,1
7,02_weekly,01_High,WEEKLY,HIGH,8
8,02_weekly,01_High,WEEKLY,MEDIUM,1
9,02_weekly,02_Medium,BI_WEEKLY,MEDIUM,5


In [165]:
df_overlap.to_clipboard()

python(89040) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [166]:
cols = ['customer', 'rr_intent_multiplier', 'taxi_consistency_rr_correction_factor',	
          'rr_regularity',	'taxi_consistency_rr_regularity_90',
          'rr_regularity_scaled',	'taxi_consistency_rr_regularity_scaled_90',
          'rr_mean_intent', 'taxi_intent_rr_actv_mean_90', 'active_days_mean',	'taxi_consistency_rr_actv_days_mean_90',
          'rr_regularity_final', 'rr_mean_intent_final',  'taxi_regularity_segment', 'taxi_intent_segment'
         ]

In [49]:
# df_merge[df_merge['customer'].isin(['62f6016b4d8a82ead77f5e81',
# '6623445c3b0cec050d82deb4',
# '5e312c6b6f27f97bc25bbdc0',
# '5d4460d68dbbed6b8548606b',
# '653cd61e41083274309b5af0'])][cols]

In [51]:
# df_merge[(df_merge['rr_regularity_final'] == '05_rare_need' )
#         &
#         (df_merge['taxi_regularity_segment'] == 'MONTHLY' )
#         &
#         (df_merge['rr_mean_intent_final'] == '01_High' )
#         &
#         (df_merge['taxi_intent_segment'] == 'MEDIUM' )
#         ][cols].head(5)

In [53]:
concatenated_df[concatenated_df['customer'] == '658d4b387315065133bbba1e']

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date,rr_mean_intent,rr_mean_intent_final,rr_intent_multiplier,rr_regularity,rr_regularity_scaled,active_days_mean,rr_regularity_final,reg_intent,city
0,658d4b387315065133bbba1e,204,210.0,2024-07-18,2023-12-28,HOOK,HOOK,UNKNOWN,21,RECENT,6,Bangalore,Low,Increasing,Low,High,Recent,Low,MALE,NPS,NPS,LINK_AUTO,LONG_DISTANCE,UNKNOWN,WEEKDAY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,658d4b387315065133bbba1e,0.0,0.0,0.0,0.0,0.0,0.0,3.0,4.0,0.0,0.0,2.0,1.0,6.0,16.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,2.0,1.0,3.0,0.0,35.0,203.0,210.0,2024-07-19,3.2,02_Medium,1.0,0.566667,0.566667,1.8,03_bi_weekly,03_bi_weekly_02_Medium,Bangalore


In [54]:
concatenated_df.shape

(480, 102)

In [55]:
concatenated_df.customer.nunique()

480